<a href="https://colab.research.google.com/github/alaaguedda/medical_report_summarization_project/blob/main/medical_summarization_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# Project setup and directories
# ===============================

import os

BASE_DIR = "/content/medical_summarization_project"

folders = [
    "data/raw",
    "data/ground_truth",
    "data/processed",
    "embeddings",
    "models/extractive",
    "models/abstractive",
    "results/extractive_summaries",
    "results/abstractive_summaries",
    "utils"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Project folders created.")


Project folders created.


In [ ]:
# ===============================
# Core NLP libraries
# ===============================

import re
import json
import numpy as np
import pandas as pd

# NLP preprocessing
import nltk
from nltk.tokenize import sent_tokenize

# Vectorization & importance
from sklearn.feature_extraction.text import TfidfVectorizer

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM
)

# Similarity & evaluation
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK resources
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# ===============================
# Load raw medical reports
# ===============================

def load_reports(path):
    reports = {}
    for file in sorted(os.listdir(path)):
        if file.endswith(".txt"):
            with open(os.path.join(path, file), "r") as f:
                reports[file] = f.read()
    return reports

raw_reports = load_reports(os.path.join(BASE_DIR, "data/raw"))
print(f"Loaded {len(raw_reports)} medical reports")


Loaded 20 medical reports


In [ ]:
# ===============================
# Text preprocessing
# ===============================

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

processed_reports = {
    k: preprocess_text(v) for k, v in raw_reports.items()
}


In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:



# ===============================
# Sentence segmentation
# ===============================

sentence_data = {}

for file, text in processed_reports.items():
    sentences = sent_tokenize(text)
    sentence_data[file] = sentences

    with open(
        os.path.join(BASE_DIR, "data/processed", file.replace(".txt", ".json")),
        "w"
    ) as f:
        json.dump(sentences, f, indent=2)

# ---- Inspection ----
print("Sentence segmentation complete.\n")

example_file = list(sentence_data.keys())[0]
example_sentences = sentence_data[example_file]

print(f"Example document: {example_file}")
print(f"Number of sentences: {len(example_sentences)}\n")

print("First 3 sentences:")
for i, s in enumerate(example_sentences[:3], 1):
    print(f"{i}. {s}")



Sentence segmentation complete.

Example document: report_01.txt
Number of sentences: 5

First 3 sentences:
1. the patient is diagnosed with diabetes mellitus after repeated high glucose readings.
2. diabetes mellitus means the body cannot properly regulate blood sugar due to insulin issues.
3. the main causes of diabetes mellitus include genetics, poor diet, and sedentary lifestyle.


In [ ]:
# ===============================
# TF-IDF importance
# ===============================

documents = list(processed_reports.values())

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(documents)
feature_names = tfidf.get_feature_names_out()

# ---- Inspection ----
print("TF-IDF matrix created.\n")
print("TF-IDF matrix shape:")
print(f"Documents: {tfidf_matrix.shape[0]}")
print(f"Vocabulary size: {tfidf_matrix.shape[1]}\n")

# Show top TF-IDF terms for one document
doc_id = 0
scores = tfidf_matrix[doc_id].toarray().flatten()
top_indices = scores.argsort()[-10:][::-1]

print("Top TF-IDF terms for first document:")
for idx in top_indices:
    print(f"{feature_names[idx]} → {scores[idx]:.4f}")


TF-IDF matrix created.

TF-IDF matrix shape:
Documents: 20
Vocabulary size: 1263

Top TF-IDF terms for first document:
diabetes mellitus → 0.4023
mellitus → 0.4023
diabetes → 0.3536
glucose → 0.1609
of diabetes → 0.1609
and sedentary → 0.0805
activity and → 0.0805
after repeated → 0.0805
cannot properly → 0.0805
consistent medication → 0.0805


In [ ]:
# ===============================
# Transformer embeddings
# ===============================

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

def get_sentence_embedding(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

sentence_embeddings = {}

for file, sentences in sentence_data.items():
    embeddings = [get_sentence_embedding(s)[0] for s in sentences]
    sentence_embeddings[file] = np.array(embeddings)

# ---- Inspection ----
example_file = list(sentence_embeddings.keys())[0]
emb = sentence_embeddings[example_file]

print("Sentence embeddings created.\n")
print(f"Example document: {example_file}")
print(f"Embedding matrix shape: {emb.shape}")
print("(num_sentences, embedding_dim)\n")

print("First sentence embedding (first 10 values):")
print(emb[0][:10])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence embeddings created.

Example document: report_01.txt
Embedding matrix shape: (5, 768)
(num_sentences, embedding_dim)

First sentence embedding (first 10 values):
[ 0.04314427 -0.15403695  0.01935874 -0.37154326  0.51791525 -0.03045896
  0.5315994   0.41613793 -0.09660314  0.05386535]


In [ ]:
# ===============================
# Sentence importance scoring
# ===============================

def sentence_importance(embeddings):
    centroid = embeddings.mean(axis=0)
    scores = cosine_similarity(embeddings, centroid.reshape(1, -1))
    return scores.flatten()

# ---- Inspection ----
example_sentences = sentence_data[example_file]
example_embeddings = sentence_embeddings[example_file]

importance_scores = sentence_importance(example_embeddings)

print("Sentence importance scores computed.\n")
print(f"Scores shape: {importance_scores.shape}\n")

# Show top-ranked sentences
ranked = sorted(
    zip(example_sentences, importance_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Top 3 most important sentences:")
for i, (s, score) in enumerate(ranked[:5], 1):
    print(f"{i}. ({score:.4f}) {s}")


Sentence importance scores computed.

Scores shape: (5,)

Top 3 most important sentences:
1. (0.9388) this patient experiences fatigue and frequent urination related to diabetes mellitus.
2. (0.9377) the patient is diagnosed with diabetes mellitus after repeated high glucose readings.
3. (0.9279) diabetes mellitus means the body cannot properly regulate blood sugar due to insulin issues.
4. (0.9274) the main causes of diabetes mellitus include genetics, poor diet, and sedentary lifestyle.
5. (0.9246) management of diabetes mellitus involves glucose monitoring, dietary control, physical activity, and consistent medication adherence.


In [ ]:
# ===============================
# Extractive summarization
# ===============================

def extractive_summary(sentences, scores, top_k=2):
    """
    Inputs:
        sentences: list[str]  -> original document sentences
        scores:    list[float] -> importance score per sentence
        top_k:     int         -> number of sentences to select

    Outputs:
        summary: str
        selected: list of tuples (sentence, score)
    """

    ranked = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )

    selected = ranked[:top_k]
    summary = " ".join([s for s, _ in selected])

    return summary, selected


# ---- Run extractive summarization ----
extractive, selected_sentences = extractive_summary(
    example_sentences,
    importance_scores,
    top_k=2
)

# ===============================
# Pretty print results
# ===============================

print("\n" + "="*60)
print("EXTRACTIVE SUMMARY RESULTS")
print("="*60 + "\n")

print("Selected Sentences (ranked by importance):\n")

for i, (sent, score) in enumerate(selected_sentences, 1):
    print(f"[{i}] Importance score: {score:.4f}")
    print(f"    Original sentence:")
    print(f"    {sent}\n")

print("-"*60)
print("FINAL EXTRACTIVE SUMMARY\n")
print(extractive)
print("="*60)



EXTRACTIVE SUMMARY RESULTS

Selected Sentences (ranked by importance):

[1] Importance score: 0.9388
    Original sentence:
    this patient experiences fatigue and frequent urination related to diabetes mellitus.

[2] Importance score: 0.9377
    Original sentence:
    the patient is diagnosed with diabetes mellitus after repeated high glucose readings.

------------------------------------------------------------
FINAL EXTRACTIVE SUMMARY

this patient experiences fatigue and frequent urination related to diabetes mellitus. the patient is diagnosed with diabetes mellitus after repeated high glucose readings.


In [ ]:
# ===============================
# Abstractive summarization
# ===============================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


In [ ]:

t5_tokenizer = AutoTokenizer.from_pretrained("t5-large")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-large")

def abstractive_summary(text, max_len=25):
    """
    Inputs:
        text: str  -> full document text
        max_len: int -> max generated tokens

    Outputs:
        summary: str
    """
    input_text = "summarize: " + text
    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True
    )
    output = t5_model.generate(**inputs, max_length=max_len)
    return t5_tokenizer.decode(output[0], skip_special_tokens=True)


# ---- Run abstractive summarization ----
original_text = processed_reports[example_file]
abstractive = abstractive_summary(original_text)


# ===============================
# Pretty print results
# ===============================

print("\n" + "="*60)
print("ABSTRACTIVE SUMMARY RESULTS")
print("="*60 + "\n")

print("ORIGINAL DOCUMENT SENTENCES\n")

for i, sent in enumerate(original_text.split(". "), 1):
    print(f"[{i}] {sent.strip()}.")

print("\n" + "-"*60)
print("GENERATED ABSTRACTIVE SUMMARY\n")
print(abstractive)
print("="*60)


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]


ABSTRACTIVE SUMMARY RESULTS

ORIGINAL DOCUMENT SENTENCES

[1] the patient is diagnosed with diabetes mellitus after repeated high glucose readings.
[2] diabetes mellitus means the body cannot properly regulate blood sugar due to insulin issues.
[3] the main causes of diabetes mellitus include genetics, poor diet, and sedentary lifestyle.
[4] this patient experiences fatigue and frequent urination related to diabetes mellitus.
[5] management of diabetes mellitus involves glucose monitoring, dietary control, physical activity, and consistent medication adherence..

------------------------------------------------------------
GENERATED ABSTRACTIVE SUMMARY

diabetes mellitus means the body cannot properly regulate blood sugar due to insulin issues . main causes include genetics


In [ ]:
# ===============================
# Load ground-truth summaries
# ===============================

ground_truth = load_reports(os.path.join(BASE_DIR, "data/ground_truth"))

# Derive ground-truth filename
gt_file = example_file.replace(".txt", "_GT.txt")
gt = ground_truth.get(gt_file, None)

print("GROUND TRUTH SUMMARY (LLM)\n")
print(gt if gt else f"No ground truth found for {gt_file}")


GROUND TRUTH SUMMARY (LLM)

Diabetes mellitus with poor glucose regulation, lifestyle causes, and management through monitoring and medication.


In [ ]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ed742101a384fb1f4ebcb5f50e203448cd13a238354eea43fbdaa7fe93c6dd99
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
# ===============================
# Evaluation (embedding + ROUGE)
# ===============================

from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
from rouge_score import rouge_scorer
import numpy as np

def evaluate_summary_all(pred, ref):
    """
    Inputs:
        pred: str (generated summary)
        ref:  str (ground-truth summary)

    Outputs:
        dict with similarity scores
    """

    # ---------- Embedding-based similarity ----------
    emb_pred = get_sentence_embedding(pred)   # shape: (1, d)
    emb_ref  = get_sentence_embedding(ref)    # shape: (1, d)

    # 1. Cosine similarity
    cosine_sim = cosine_similarity(emb_pred, emb_ref)[0][0]

    # 2. Euclidean distance → similarity
    euclid_dist = euclidean_distances(emb_pred, emb_ref)[0][0]
    euclid_sim = 1 / (1 + euclid_dist)

    # 3. Manhattan distance → similarity
    manhattan_dist = manhattan_distances(emb_pred, emb_ref)[0][0]
    manhattan_sim = 1 / (1 + manhattan_dist)

    # ---------- ROUGE scores ----------
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    rouge_scores = scorer.score(ref, pred)

    return {
        # Embedding metrics
        "cosine_similarity": cosine_sim,
        "euclidean_similarity": euclid_sim,
        "manhattan_similarity": manhattan_sim,

        # ROUGE (F1 scores)
        "rouge1_f1": rouge_scores["rouge1"].fmeasure,
        "rouge2_f1": rouge_scores["rouge2"].fmeasure,
        "rougeL_f1": rouge_scores["rougeL"].fmeasure
    }
# ---- Run evaluation ----
if gt:
    extractive_scores = evaluate_summary_all(extractive, gt)
    abstractive_scores = evaluate_summary_all(abstractive, gt)

    print("Similarity to ground truth\n")

    print("EXTRACTIVE SUMMARY")
    for k, v in extractive_scores.items():
        print(f"{k}: {v:.4f}")

    print("\nABSTRACTIVE SUMMARY")
    for k, v in abstractive_scores.items():
        print(f"{k}: {v:.4f}")


Similarity to ground truth

EXTRACTIVE SUMMARY
cosine_similarity: 0.8495
euclidean_similarity: 0.1619
manhattan_similarity: 0.0088
rouge1_f1: 0.2703
rouge2_f1: 0.0571
rougeL_f1: 0.2162

ABSTRACTIVE SUMMARY
cosine_similarity: 0.8391
euclidean_similarity: 0.1526
manhattan_similarity: 0.0082
rouge1_f1: 0.2424
rouge2_f1: 0.0645
rougeL_f1: 0.2424


# **TRAINED MODEL**

In [1]:
!pip install kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aminexdr/bhc-mimic-iv-summary
!unzip bhc-mimic-iv-summary.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aminexdr/bhc-mimic-iv-summary
License(s): unknown
 66% 293M/446M [00:00<00:00, 3.07GB/s]
100% 446M/446M [00:00<00:00, 2.96GB/s]
Archive:  bhc-mimic-iv-summary.zip
  inflating: BHC_MIMIC-IV.csv        


In [2]:
import pandas as pd

df = pd.read_csv("BHC_MIMIC-IV.csv")


In [3]:
df = df[['input', 'target']].dropna()

# Shuffle first
df = df.sample(frac=1, random_state=42)

# Take only 10,000 samples
df = df.iloc[:10000]

df.shape

(10000, 2)

In [4]:
df.isnull().sum()
df = df.dropna()

In [5]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [6]:
!pip install transformers datasets evaluate rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=7f9b45d771a896360d383d719be0696403998ec7db790608d7ab51721ea87b7e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [7]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [8]:
max_input_length = 256
max_target_length = 64

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["input"]]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        examples["target"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [10]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,   # Reduced from 8 to be safe
    eval_accumulation_steps=10,     # <--- MOVE DATA TO CPU EVERY 10 STEPS
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    logging_steps=200,
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

ValueError: fp16 mixed precision requires a device in ('xpu', 'cuda', 'npu', 'xla', 'mlu', 'musa', 'hpu', 'sdaa', 'mps') (not 'xla').

In [ ]:
trainer.train()

In [ ]:
import torch
import gc

# 1. Clear Python garbage collector
gc.collect()

# 2. Clear NVIDIA cache
torch.cuda.empty_cache()

# 3. Now run the evaluation
results = trainer.evaluate()

In [ ]:
!pip install evaluate rouge_score absl-py

In [ ]:
!pip install bert-score nltk

In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")
meteor = evaluate.load("meteor")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Remove empty strings
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # ROUGE
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # BLEU (needs tokenized format)
    bleu_result = bleu.compute(
        predictions=[pred.split() for pred in decoded_preds],
        references=[[label.split()] for label in decoded_labels]
    )

    # METEOR
    meteor_result = meteor.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # BERTScore
    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="en"
    )

    return {
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
        "BLEU": bleu_result["bleu"],
        "METEOR": meteor_result["meteor"],
        "BERTScore-F1": np.mean(bert_result["f1"])
    }

results = trainer.evaluate()

print("\n========== MODEL EVALUATION RESULTS ==========\n")

print(f"ROUGE-1   : {results['eval_ROUGE-1']:.4f}")
print("  → Measures word overlap (unigram recall)\n")

print(f"ROUGE-2   : {results['eval_ROUGE-2']:.4f}")
print("  → Measures bigram overlap (captures fluency)\n")

print(f"ROUGE-L   : {results['eval_ROUGE-L']:.4f}")
print("  → Measures longest common subsequence\n")

print(f"BLEU      : {results['eval_BLEU']:.4f}")
print("  → Measures precision of generated words\n")

print(f"METEOR    : {results['eval_METEOR']:.4f}")
print("  → Considers synonym matching and alignment\n")

print(f"BERTScore : {results['eval_BERTScore-F1']:.4f}")
print("  → Measures deep semantic similarity\n")

print("==============================================")

In [ ]:
import torch

def generate_summary(text):
    device = model.device  # automatically detect cuda or cpu

    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,   # match your training max length
        truncation=True
    )

    # Move inputs to same device as model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_length=64,   # match your training target length
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
generate_summary(df['input'].iloc[0])